# Predicting Car Insurance Claim Frequencies using French Motor Dataset

## Importing Libraries

In [12]:
from sklearn.datasets import fetch_openml
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Importing Dataset

In [7]:
# Direct CSV URL from Hugging Face dataset repository
url_freq = "https://huggingface.co/datasets/mabilton/fremtpl2/resolve/main/freMTPL2freq.csv"
url_sev = "https://huggingface.co/datasets/mabilton/fremtpl2/resolve/main/freMTPL2sev.csv"

try:
    # Load frequency dataset
    df_freq = pd.read_csv(url_freq)
    print("Frequency dataset loaded:", df_freq.shape)

    # Load severity dataset
    df_sev = pd.read_csv(url_sev)
    print("Severity dataset loaded:", df_sev.shape)
except Exception as e:
    print("Error loading dataset:", e)

Frequency dataset loaded: (678013, 12)
Severity dataset loaded: (26639, 2)


## Claim Frequency


In [47]:
print("--- FREQUENCY DATA ---")
print('Exploring Frequency Data')
df_freq.head()

print("Checking if any columns have NULL Values")
df_freq.isnull().sum()

df_freq.info()

df_freq.describe()

df_freq.dtypes

--- FREQUENCY DATA ---
Exploring Frequency Data
Checking if any columns have NULL Values
<class 'pandas.DataFrame'>
RangeIndex: 678013 entries, 0 to 678012
Data columns (total 11 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   ClaimNb     678013 non-null  int64  
 1   Exposure    678013 non-null  float64
 2   VehPower    678013 non-null  int64  
 3   VehAge      678013 non-null  int64  
 4   DrivAge     678013 non-null  int64  
 5   BonusMalus  678013 non-null  int64  
 6   VehBrand    678013 non-null  str    
 7   VehGas      678013 non-null  str    
 8   Area        678013 non-null  str    
 9   Density     678013 non-null  int64  
 10  Region      678013 non-null  str    
dtypes: float64(1), int64(6), str(4)
memory usage: 56.9 MB


ClaimNb         int64
Exposure      float64
VehPower        int64
VehAge          int64
DrivAge         int64
BonusMalus      int64
VehBrand          str
VehGas            str
Area              str
Density         int64
Region            str
dtype: object

## Exploring by plotting

### What does each column represent?
*   ID Pol - unique random number assigned to the driver

*   Claim Nb - how many times this specific person filed a car insurance claim during the year

*   Exposure - This is the fraction of the year the policy was actually active.
    *   If Exposure is 1.0, they were insured for the full year.
    *   If Exposure is 0.5, they were only insured for six months.
    *   Why it matters: It is not fair to compare a guy who didn't crash in 1 month to a guy who didn't crash in 12 months. Exposure levels the playing field.

*   VehPower - A categorized number representing the engine's horsepower. A high number means a faster, more powerful car (which might encourage riskier driving).

*   VehAge - How old the car is in years. People might drive a brand-new car very cautiously, while they might not care if they scratch a 15-year-old beater.

*   DrivAge - The age of the primary driver in years. This helps us see the difference between a chaotic 18-year-old and an experienced 45-year-old.

*   BonusMalus - This is the French safe-driving score.
    *   If you drive safely year after year, your score goes down (a "Bonus"). The best possible score is usually around 50.
    *   If you cause accidents, your score goes up (a "Malus"). A score over 100 means you are penalized and paying extra.
*   VehBrand - The brand of the car. In this dataset, the real names (like Renault or Peugeot) are hidden and replaced with codes like B12 or B01

*   VehGas - The type of fuel the car takes—Diesel or Regular. (In Europe, diesel cars are often used for longer commutes, which might mean more time on the road)

*   Area - A letter code (A through F) grouping geographic areas by how rural or urban they are.

*   Density - The population density of the city the driver lives in (people per square kilometer). This is a pure numbers game: more people = more cars = more chances to bump into someone.

*   Region - The specific administrative region in France where the policyholder lives (e.g., Brittany, Ile-de-France). This can capture regional weather patterns, road conditions, or local driving cultures.

In [19]:
df_freq = df_freq.drop(columns=['IDpol'])

In [48]:
numeric_cols = df_freq.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = df_freq.select_dtypes(include=['object','str']).columns
print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

Numeric columns: Index(['ClaimNb', 'Exposure', 'VehPower', 'VehAge', 'DrivAge', 'BonusMalus',
       'Density'],
      dtype='str')
Categorical columns: Index(['VehBrand', 'VehGas', 'Area', 'Region'], dtype='str')


In [52]:
df_freq.describe(include = 'all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ClaimNb,678013.0,NaN,NaN,NaN,0.053247,0.240117,0.0,0.0,0.0,0.0,16.0
Exposure,678013.0,NaN,NaN,NaN,0.52875,0.364442,0.002732,0.18,0.49,0.99,2.01
VehPower,678013.0,NaN,NaN,NaN,6.454631,2.050906,4.0,5.0,6.0,7.0,15.0
VehAge,678013.0,NaN,NaN,NaN,7.044265,5.666232,0.0,2.0,6.0,11.0,100.0
DrivAge,678013.0,NaN,NaN,NaN,45.499122,14.137444,18.0,34.0,44.0,55.0,100.0
BonusMalus,678013.0,NaN,NaN,NaN,59.761502,15.636658,50.0,50.0,50.0,64.0,230.0
VehBrand,678013,11,B12,166024,NaN,NaN,NaN,NaN,NaN,NaN,NaN
VehGas,678013,2,Regular,345877,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Area,678013,6,C,191880,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Density,678013.0,NaN,NaN,NaN,1792.422405,3958.646564,1.0,92.0,393.0,1658.0,27000.0


In [44]:
df_freq.select_dtypes(include='number').describe().T

,count,mean,std,min,25%,50%,75%,max
ClaimNb,678013.0,0.053247,0.240117,0.000000,0.00,0.00,0.00,16.00
Exposure,678013.0,0.528750,0.364442,0.002732,0.18,0.49,0.99,2.01
VehPower,678013.0,6.454631,2.050906,4.000000,5.00,6.00,7.00,15.00
VehAge,678013.0,7.044265,5.666232,0.000000,2.00,6.00,11.00,100.00
DrivAge,678013.0,45.499122,14.137444,18.000000,34.00,44.00,55.00,100.00
BonusMalus,678013.0,59.761502,15.636658,50.000000,50.00,50.00,64.00,230.00
Density,678013.0,1792.422405,3958.646564,1.000000,92.00,393.00,1658.00,27000.00


In [49]:
df_freq.select_dtypes(include=['object', 'category','str']).describe().T

,count,unique,top,freq
VehBrand,678013,11,B12,166024
VehGas,678013,2,Regular,345877
Area,678013,6,C,191880
Region,678013,21,Centre,160601


In [53]:
df_freq['VehAge'].value_counts().sort_index().tail(20)

VehAge
62      1
63      1
64      1
65      2
66      2
68      2
69      4
70      3
71      1
76      1
78      1
79      1
80      3
81      3
82      1
83      2
84      1
85      1
99     23
100    25
Name: count, dtype: int64